# FeedbackGAN — Drug Discovery on Google Colab

**Train-from-scratch** version for KOR-targeted molecule generation.

**You have:** the KOR dataset (`data_clean_kop.csv`)
**You will train:** Autoencoder → Predictor → WGAN-GP → FeedbackGAN

---

### ⚠️ Read first
1. **Runtime → Change runtime type → GPU** (T4 is fine).
2. This code targets **TensorFlow 2.x with Keras 2**. Modern Colab ships Keras 3,
   so Section 2 installs a compatible stack and forces the Keras 2 API via
   `TF_USE_LEGACY_KERAS=1`. **Run Section 2, then Runtime → Restart session, then
   continue from Section 1.**
3. Have `data_clean_kop.csv` ready (`smiles,pIC50` columns).

Smoke-test epochs are tiny on purpose so the whole pipeline runs in ~10 min. Raise
them (comments mark each) for real results.

## 0. First: install (Section 2), restart, then run from Section 1

If you've already installed + restarted, just continue. Otherwise jump to
Section 2, run it, restart the session, and come back here.

In [ ]:
# Quick GPU + Keras sanity check (run AFTER installing + restarting)
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
print("TF version:", tf.__version__)
try:
    print("Keras version:", tf.keras.__version__, "(want 2.x)")
except Exception:
    pass
gpus = tf.config.list_physical_devices('GPU')
print("GPUs:", gpus)
if not gpus:
    print("\n⚠️  No GPU detected! Runtime → Change runtime type → GPU")
else:
    print("\n✅ GPU ready")

## 1. Clone your code from GitHub

Replace the URL with your repo. We `cd` into it so relative paths work.

In [ ]:
# 👇 EDIT THIS to your repo
REPO_URL = "https://github.com/0x41hd/GAN-Drug-Generator.git"
REPO_DIR = "GAN-Drug-Generator"

import os
if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL
else:
    print("Repo already cloned")

%cd $REPO_DIR
!ls -la

## 2. Install dependencies (RUN FIRST, then restart)

This installs a Keras-2 compatible stack. `bunch` is replaced by `munch`
(maintained); the code falls back to it automatically.

In [ ]:
!pip install -q "tensorflow==2.15.1" "tf-keras" "rdkit" "munch" "seaborn" "tqdm" "scikit-learn" "numpy<2.0"

import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"  # force Keras 2 API

import tensorflow as tf, numpy as np, rdkit
print("TF:", tf.__version__)
print("Keras:", tf.keras.__version__)
print("NumPy:", np.__version__)
print("RDKit:", rdkit.__version__)
print("\n✅ If Keras shows 2.15.x, you're good.")
print("⚠️  If pip changed TensorFlow, do Runtime → Restart session, then run from Section 1.")

## 3. Provide the dataset

If `data/data_clean_kop.csv` isn't already in the repo, upload it. Format:
```
smiles,pIC50
COc1ccc2c(c1)CC(N)C2,6.3
```

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.makedirs('data', exist_ok=True)

if os.path.exists('data/data_clean_kop.csv'):
    print("✅ Dataset already present")
else:
    print("Upload your data_clean_kop.csv now:")
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded:
        os.rename(fname, 'data/data_clean_kop.csv')
    print("✅ Saved to data/data_clean_kop.csv")

import pandas as pd
df = pd.read_csv('data/data_clean_kop.csv')
print("\nShape:", df.shape)
print(df.head())

## 4. Build Vocabulary and config files

Generates `Vocab_complete.txt`, `configReinforce.json`, `configPredictor.json`.

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import pandas as pd
from Vocabulary2 import Vocabulary

df = pd.read_csv('data/data_clean_kop.csv')
all_smiles = df[df.columns[0]].astype(str).tolist()

vocab_tmp = Vocabulary('Vocab_complete.txt', max_len=70)
vocab_tmp.update_vocab(all_smiles)
print("✅ Vocab_complete.txt written, vocab size =", vocab_tmp.vocab_size)

In [ ]:
import json

config_reinforce = {
    "smile_len_threshold": 65,
    "exp_name": "feedbackGAN_kor",
    "datapath_kor": "data/data_clean_kop.csv"
}
with open('configReinforce.json', 'w') as f:
    json.dump(config_reinforce, f, indent=2)

config_predictor = {
    "smile_len_threshold": 65, "exp_name": "predictor_kor", "input_length": 70,
    "lr": 0.001, "beta_1": 0.9, "beta_2": 0.999, "batch_size": 64,
    "epochs_dense": 30, "loss_criterium": "mse", "activation_dense": "linear",
    "rnn": "LSTM", "n_units": 128, "dropout": 0.3, "epochs": 30,
    "model_name": "kor_predictor", "checkpoint_dir": "predictor_weigths_kor/"
}
with open('configPredictor.json', 'w') as f:
    json.dump(config_predictor, f, indent=2)

print("✅ configReinforce.json and configPredictor.json written")

## 5. Get the SA-score fragment file

`fpscores.pkl.gz` is needed for synthetic-accessibility scoring.

In [ ]:
import os
if not os.path.exists('fpscores.pkl.gz'):
    !wget -q https://raw.githubusercontent.com/rdkit/rdkit/master/Contrib/SA_Score/fpscores.pkl.gz
    print("✅ Downloaded fpscores.pkl.gz")
else:
    print("✅ fpscores.pkl.gz already present")

## 6. Create directory structure

In [ ]:
import os
for d in ['AE/scratch_ae/', 'GAN_model/GAN_weights/', 'predictor_weigths_kor/']:
    os.makedirs(d, exist_ok=True)
print("✅ Folders created")

## 7. Train the Autoencoder

> Smoke test: `EPOCHS_AE = 2`. Real run: 80–100.

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import numpy as np
from vocabulary import Vocabulary
from autoencoder import Autoencoder as AE

EPOCHS_AE = 2          # ⬅️ raise to ~80-100 for real training
BATCH_AE  = 128
AE_SAVE_PATH = 'AE/scratch_ae/'

import pandas as pd
df = pd.read_csv('data/data_clean_kop.csv')
smiles_raw = df[df.columns[0]].astype(str).tolist()

vocab = Vocabulary('Vocab_complete.txt', max_len=70)
tok, _ = vocab.tokenize(smiles_raw)

X_train = vocab.encode(tok)
n = len(X_train)
X_train = np.array(X_train)  # shape (n, max_len) — encoder Input is (None,)
X2_train = vocab.one_hot_encoder(tok)
Y_train  = vocab.get_target(X2_train, 'OHE')

print("Training samples:", n)
print("X_train:", X_train.shape, " X2_train:", X2_train.shape, " Y_train:", Y_train.shape)

latent_dim = 256; lstm_units = 512; batch_norm = True
batch_norm_momentum = 0.9; noise_std = 0.1; numb_dec_layer = 2; emb_dim = 256
decoder_input_shape = (vocab.max_len, vocab.vocab_size)
output_dim = vocab.vocab_size

autoencoder = AE(AE_SAVE_PATH, decoder_input_shape, latent_dim, lstm_units,
                 output_dim, batch_norm, batch_norm_momentum, noise_std,
                 numb_dec_layer, emb_dim, vocab.vocab_size, vocab.max_len)

autoencoder.fit_model(X_train, X2_train, Y_train, EPOCHS_AE, BATCH_AE, 'adam')
print("\n✅ Autoencoder trained. Weights in", AE_SAVE_PATH)
!ls -la $AE_SAVE_PATH

Pick the best checkpoint (lowest val_loss).

In [ ]:
import glob
ae_ckpts = sorted(glob.glob('AE/scratch_ae/model--*.weights.h5'))
print("Autoencoder checkpoints:")
for c in ae_ckpts:
    print("  ", c)
AE_WEIGHTS = ae_ckpts[-1] if ae_ckpts else None
print("\nUsing:", AE_WEIGHTS)

## 8. Train the Predictor ensemble (5 LSTM models)

Estimates KOR pIC50 from SMILES. Saves to `predictor_weigths_kor/`.

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import numpy as np
import pandas as pd
from Vocabulary2 import Vocabulary
from utils import load_config, reading_csv
from predictor import Predictor

config = load_config('configPredictor.json', 'kor')
vocab = Vocabulary('Vocab_complete.txt', max_len=70)

smiles, labels = reading_csv(config, 'kor')
print("Predictor training molecules:", len(smiles))

tok, og_idx = vocab.tokenize(smiles)
X = np.asarray(vocab.encode(tok))
y = np.asarray([labels[i] for i in og_idx], dtype=float)

from sklearn.model_selection import train_test_split
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42)

# predictor.train_model expects: [X_train, y_train, X_test, y_test, X_val, y_val]
data = [X_tr, y_tr, X_te, y_te, X_val, y_val]
print("Train/Val/Test:", X_tr.shape, X_val.shape, X_te.shape)

In [ ]:
predictor = Predictor(config, vocab, 'dnn', 'SMILES', 'kor', load=False)

import os
os.makedirs('predictor_weigths_kor', exist_ok=True)

for i in range(5):
    print(f"\n=== Training predictor model {i} ===")
    predictor.model = predictor.loaded_models[i]
    predictor.train_model(data, searchParam=False)
    # Keras 3 requires '.weights.h5'; load_models() also accepts legacy '.hdf5'.
    save_path = f"predictor_weigths_kor/kormodel{i}.weights.h5"
    predictor.loaded_models[i].save_weights(save_path)
    print(f"✅ Saved {save_path}")

print("\n✅ All 5 predictor models trained")
!ls -la predictor_weigths_kor/

## 9. Train the base WGAN-GP

> Smoke test: `EPOCHS_GAN = 50`. Real: 5,000–10,000.

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import numpy as np
from wgan import WGANGP

import pandas as pd
df = pd.read_csv('data/data_clean_kop.csv')
smiles_raw = df[df.columns[0]].astype(str).tolist()

tok, _ = vocab.tokenize(smiles_raw)
X_enc = np.array(vocab.encode(tok))  # (n, max_len) — matches encoder Input(None,)
n = len(X_enc)

autoencoder.load_autoencoder_model(AE_WEIGHTS)
x_latent = autoencoder.smiles_to_latent_model.predict(X_enc)
print("Latent data shape:", x_latent.shape)

input_dim = 256
critic_layers_units = [256, 256, 256]
critic_lr = 0.0001; gp_weight = 10; z_dim = 64
generator_layers_units = [128, 256, 256, 256, 256]
generator_batch_norm_momentum = 0.9; generator_lr = 0.0001; batch_size = 64
critic_optimizer = 'adam'; generator_optimizer = 'adam'
critic_dropout = 0.2; generator_dropout = 0.2

gan = WGANGP(input_dim, critic_layers_units, critic_lr, critic_dropout, gp_weight,
             z_dim, generator_layers_units, generator_batch_norm_momentum,
             generator_lr, generator_dropout, batch_size,
             critic_optimizer, generator_optimizer)
print("✅ GAN built")

In [ ]:
import os
EPOCHS_GAN = 50      # ⬅️ raise to ~5000-10000 for real training
BASE_RUN = 'GAN_model/base_run/'
for sub in ['viz', 'weights', 'generated_data']:
    os.makedirs(BASE_RUN + sub, exist_ok=True)

gan.autoencoder = autoencoder
gan.vocab = vocab

gan.train(x_latent, batch_size, EPOCHS_GAN, BASE_RUN, autoencoder, vocab,
          print_every_n_epochs=10, critic_loops=5)

gan.critic.save_weights('GAN_model/GAN_weights/critic_weights-10000.weights.h5')
gan.generator.save_weights('GAN_model/GAN_weights/generator_weights-10000.weights.h5')
print("\n✅ Base GAN trained, weights saved to GAN_model/GAN_weights/")

## 10. Run the FeedbackGAN loop

Each epoch: generate → score with predictor → replace weakest training samples
with best generated, pushing toward higher predicted pIC50.

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import numpy as np, csv
from utils import evaluate_property, plot_hist_both

n_run = 'run_feedback_colab'
run_folder = 'GAN_model/' + n_run + '/'
for sub in ['', 'viz', 'generated_data', 'weights', 'feedbackGAN']:
    os.makedirs(os.path.join(run_folder, sub), exist_ok=True)

gan.epoch = 0
gan.autoencoder = autoencoder
gan.vocab = vocab

valid_before, perc_before, _ = gan.sample_valid_data(200, run_folder, True)
pred_before, _ = evaluate_property(predictor, valid_before, 'kor')
print("Valid % before:", perc_before)

EPOCHS_FB = 50       # ⬅️ raise to 500 for real training
n_to_generate = 200
threshold = 5
info = 'max'

gan.train_feedbackGAN(x_latent, smiles_raw, batch_size, EPOCHS_FB, run_folder,
                      autoencoder, vocab, predictor, n_to_generate, threshold,
                      info, 'kor', print_every_n_epochs=10, critic_loops=5)

valid_after, perc_after, _ = gan.sample_valid_data(200, run_folder, True)
pred_after, _ = evaluate_property(predictor, valid_after, 'kor')
print("\nValid % after:", perc_after)

## 11. Compare before vs after

In [ ]:
import numpy as np
pred_before = np.array(pred_before).astype(float)
pred_after  = np.array(pred_after).astype(float)

print("Mean pIC50 before:", pred_before.mean())
print("Mean pIC50 after :", pred_after.mean())
print("Shift:", pred_after.mean() - pred_before.mean())

diff = plot_hist_both(pred_before, pred_after, 'kor')
print("\nDifference between averages:", diff)

In [ ]:
import csv, os
with open(os.path.join(run_folder, 'generated_molecules.csv'), 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['SMILES'])
    for s in valid_after:
        writer.writerow([s])
print("✅ Saved generated_molecules.csv")

import pandas as pd
out = pd.DataFrame({'SMILES': valid_after[:len(pred_after)], 'pred_pIC50': pred_after})
out = out.sort_values('pred_pIC50', ascending=False)
out.head(15)

## 12. (Optional) Save results to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
dest = '/content/drive/MyDrive/FeedbackGAN_results/'
os.makedirs(dest, exist_ok=True)

for src in ['AE/scratch_ae', 'GAN_model/GAN_weights', 'predictor_weigths_kor',
            'GAN_model/run_feedback_colab']:
    if os.path.exists(src):
        d = os.path.join(dest, src.replace('/', '_'))
        shutil.copytree(src, d, dirs_exist_ok=True)
        print("Copied", src, "->", d)
print("\n✅ Saved to Google Drive")

---
## Troubleshooting

**Small epochs on purpose.** `EPOCHS_AE=2`, `EPOCHS_GAN=50`, `EPOCHS_FB=50` are
smoke-test values. Raise them (comments mark each) for real results.

**"Embedding index out of range":** vocab mismatch. Re-run Section 4 and make sure
the same `vocab` object flows into autoencoder, predictor, and GAN.

**Out of GPU RAM:** lower `batch_size` from 64 to 32.

**Slow `sample_valid_data` at epoch 0:** normal — an untrained generator makes few
valid SMILES. The MAX_ATTEMPTS guard prevents an infinite loop.

**Keras errors / "filepath must end in .keras":** you're on Keras 3. Re-run
Section 2 and restart the session so `TF_USE_LEGACY_KERAS=1` takes effect.

---
*Built for [0x41hd](https://github.com/0x41hd)*